### Random Noise Environment에서 넣을 것

최소 2개.

| 종류                        | 목적    |
| ------------------------- | ----- |
| Gaussian amplitude noise  | 세기 왜곡 |
| Random phase perturbation | 위상 왜곡 |


In [2]:
import pickle
import os
import sys
from tqdm import tqdm
import torch
import torch.nn as nn
import logging
import csv
from torch.utils.data import DataLoader, random_split
import gc

from Utils.dataset import MMFiDataset
from Utils.SionnaPerturbation import SionnaPerturbation

# =====================================================================
# 1. 데이터 로드
# =====================================================================

with open("perturbation_dataset_v1.pkl", "rb") as f:
    perturb_data = pickle.load(f)

target_actions = [
    "A01",
    "A12",
    "A08",
    "A27"
]

full_dataset = MMFiDataset(
    env_dirs=["./E01", "./E02", "./E03"],
    target_actions=target_actions,
    perturbation=SionnaPerturbation(
        perturb_data
    )
)

sample_csi, sample_pose = full_dataset[0]

print(sample_csi.shape)
print(sample_pose.shape)

# =====================================================================
# 재현 가능한 Train/Val Split
# =====================================================================

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

generator = torch.Generator().manual_seed(42)

train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size],
    generator=generator
)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=0
)

print(f"🔥 Train: {len(train_dataset)}")
print(f"🔥 Val: {len(val_dataset)}")

# =====================================================================
# 2. 모델 로드
# =====================================================================

sys.path.append(os.path.abspath("DT-Pose"))

from model.model import MAE_Encoder, ViT_Pose_Decoder

encoder = MAE_Encoder(
    image_size=(114, 30),
    patch_size=(2, 2),
    emb_dim=256,
    input_dim=2
)

model = ViT_Pose_Decoder(
    encoder=encoder,
    keypoints=17,
    coor_num=3,
    dataset='mmfi-csi'
).cuda()

print("✅ 모델 CUDA 로딩 완료!")

# =====================================================================
# 3. Logging
# =====================================================================

save_dir = "./checkpoints_fusion"

os.makedirs(save_dir, exist_ok=True)

log_file = os.path.join(
    save_dir,
    "sionna_training_E01_E03.log"
)

loss_csv = os.path.join(
    save_dir,
    "sionna_loss_history_E01_E03.csv"
)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler(log_file),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger()

# =====================================================================
# 4. Optimizer
# =====================================================================

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

criterion = nn.MSELoss()

best_val_loss = float('inf')

# =====================================================================
# Resume
# =====================================================================

resume_path = "./checkpoints_fusion/sionna_checkpoint_epoch_E01_E03150.pth"

start_epoch = 0

if os.path.exists(resume_path):

    checkpoint = torch.load(
        resume_path,
        map_location="cpu"
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    start_epoch = checkpoint["epoch"]

    print("✅ Resume Success")
    print(f"✅ Resume Epoch: {start_epoch}")

else:

    print("⚠️ Checkpoint Not Found")

# =====================================================================
# Train
# =====================================================================

logger.info(
    f"🔥 Resume Epoch {start_epoch} -> 200 | "
    f"Train: {len(train_dataset)} | "
    f"Val: {len(val_dataset)}"
)

val_interval = 10
last_val_loss = float('inf')

for epoch in range(start_epoch, 200):

    model.train()

    total_train_loss = 0

    train_pbar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1:03d}/200",
        leave=False
    )

    for batch_idx, (csi_batch, pose_batch) in enumerate(train_pbar):

        try:

            csi_batch = csi_batch.cuda()
            pose_batch = pose_batch.cuda()

            csi_batch = csi_batch.view(-1, 2, 114, 30)
            pose_batch = pose_batch.view(-1, 17, 3)

            optimizer.zero_grad()

            output = model(csi_batch)

            predicted_pose = (
                output[0]
                if isinstance(output, tuple)
                else output
            )

            predicted_pose = predicted_pose.view(
                -1,
                17,
                3
            )

            loss = criterion(
                predicted_pose,
                pose_batch
            )

            if torch.isnan(loss):
                raise ValueError(
                    f"NaN Loss 발생! batch={batch_idx}"
                )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()

            total_train_loss += loss.item()

        except Exception as e:

            logger.error(
                f"❌ Epoch {epoch+1}, Batch {batch_idx}: {e}"
            )

            torch.save(
                {
                    'csi': csi_batch,
                    'pose': pose_batch
                },
                "error_batch.pt"
            )

            raise e

    avg_train_loss = total_train_loss / len(train_loader)

    # ==================================================
    # Validation
    # ==================================================

    if epoch == start_epoch or (epoch + 1) % val_interval == 0:

        model.eval()

        total_val_loss = 0

        with torch.no_grad():

            for csi_batch, pose_batch in val_loader:

                csi_batch = csi_batch.cuda()
                pose_batch = pose_batch.cuda()

                csi_batch = csi_batch.view(
                    -1,
                    2,
                    114,
                    30
                )

                pose_batch = pose_batch.view(
                    -1,
                    17,
                    3
                )

                output = model(csi_batch)

                predicted_pose = (
                    output[0]
                    if isinstance(output, tuple)
                    else output
                )

                predicted_pose = predicted_pose.view(
                    -1,
                    17,
                    3
                )

                val_loss = criterion(
                    predicted_pose,
                    pose_batch
                )

                total_val_loss += val_loss.item()

        avg_val_loss = (
            total_val_loss /
            len(val_loader)
        )

        last_val_loss = avg_val_loss

        logger.info(
            f"Epoch [{epoch+1:03d}/200] | "
            f"Train Loss: {avg_train_loss:.5f} | "
            f"Val Loss: {avg_val_loss:.5f}"
        )

        if avg_val_loss < best_val_loss:

            best_val_loss = avg_val_loss

            torch.save(
                model.state_dict(),
                os.path.join(
                    save_dir,
                    "sionna_best_model_E01_E03.pth"
                )
            )

            logger.info(
                f"✨ Best Model Updated "
                f"(Val={avg_val_loss:.5f})"
            )

    else:

        logger.info(
            f"Epoch [{epoch+1:03d}/200] | "
            f"Train Loss: {avg_train_loss:.5f}"
        )

    with open(
        loss_csv,
        'a',
        newline=''
    ) as f:

        csv.writer(f).writerow(
            [
                epoch + 1,
                avg_train_loss,
                last_val_loss
            ]
        )

    gc.collect()
    torch.cuda.empty_cache()

    # ==================================================
    # Checkpoint
    # ==================================================

    if (epoch + 1) % 30 == 0:

        torch.save(
            {
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict()
            },
            os.path.join(
                save_dir,
                f"sionna_checkpoint_epoch_E01_E03{epoch+1}.pth"
            )
        )

logger.info("🎉 학습 완료!")

📂 MMFi Dataset 스캔 시작
▶️ Scan: ./E01
▶️ Scan: ./E02
▶️ Scan: ./E03
✅ 총 샘플 수: 35640
🔥 Noise Applied | idx=0
torch.Size([6840])
torch.Size([51])
🔥 Train: 28512
🔥 Val: 7128


c:\Users\yujaerim\miniconda3\envs\deepNet\lib\site-packages\timm\models\layers\__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
2026-06-08 11:16:37,958 [INFO] 🔥 Resume Epoch 150 -> 200 | Train: 28512 | Val: 7128


✅ 모델 CUDA 로딩 완료!
✅ Resume Success
✅ Resume Epoch: 150


Epoch 151/200:   1%|          | 2/223 [00:01<02:48,  1.31it/s]c:\Users\yujaerim\OneDrive\Desktop\coputerVisonDNN\Utils\SionnaPerturbation.py:27: RuntimeWarning: overflow encountered in multiply
  csi[0] *= factor
Epoch 151/200:  32%|███▏      | 72/223 [00:45<01:34,  1.59it/s]

🔥 Noise Applied | idx=2


...

Epoch 200/200:  83%|████████▎ | 185/223 [01:52<00:23,  1.65it/s]

🔥 Noise Applied | idx=0


Epoch 200/200:  88%|████████▊ | 196/223 [01:58<00:16,  1.65it/s]

🔥 Noise Applied | idx=1


2026-06-08 13:14:38,221 [INFO] Epoch [200/200] | Train Loss: 0.01697 | Val Loss: 0.01690
2026-06-08 13:14:38,270 [INFO] ✨ Best Model Updated (Val=0.01690)
2026-06-08 13:14:38,392 [INFO] 🎉 학습 완료!


In [1]:
import torch

ckpt = torch.load(
    "checkpoints_fusion\sionna_checkpoint_epoch_E01_E03150.pth",
    map_location="cpu"
)

print(type(ckpt))

if isinstance(ckpt, dict):
    print(ckpt.keys())

<class 'dict'>
dict_keys(['epoch', 'model_state_dict'])
